# Novel Improvement

## Class Imbalance

### 1. Import Packages and load preprocessed Dataset

In [1]:
#Import an Load the preprocessed DataSet
from pathlib import Path
import sys
import time
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, balanced_accuracy_score, confusion_matrix
)

if Path.cwd().name == "notebooks":
    PROJECT_ROOT = Path.cwd().parent
else:
    PROJECT_ROOT = Path.cwd()

data_dir = PROJECT_ROOT / "data"

X_train = pd.read_csv(data_dir / "X_train_preprocessed.csv")
X_test = pd.read_csv(data_dir / "X_test_preprocessed.csv")
y_train = pd.read_csv(data_dir / "y_train_preprocessed.csv").squeeze()
y_test = pd.read_csv(data_dir / "y_test_preprocessed.csv").squeeze()

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train distribution:\n", y_train.value_counts())

X_train shape: (222960, 35)
X_test shape: (55741, 35)
y_train distribution:
 heart_disease_target
False    218247
True       4713
Name: count, dtype: int64


### 2. Importing the eLCS implementation

In [2]:
#Import eLCS

ELCS_ROOT = PROJECT_ROOT / "external" / "scikit-eLCS-master"

if not ELCS_ROOT.exists():
    raise FileNotFoundError(f"Expected the supplied eLCS package at: {ELCS_ROOT}")

if str(ELCS_ROOT) not in sys.path:
    sys.path.insert(0, str(ELCS_ROOT))

from skeLCS import eLCS

### 3. Train the model with & without class imbalance

Train the model on the 10 seeds use for the baseline and produce the measurements (accurancy, precision, recall, F1, balance accuracy, ROC-AUC, PR-AUC, traning seconds, and confusion matrix)

In [3]:
from imblearn.over_sampling import SMOTE
from sklearn.metrics import roc_auc_score, average_precision_score

X_train_array = X_train.to_numpy()
X_test_array = X_test.to_numpy()
y_train_array = y_train.to_numpy()
y_test_array = y_test.to_numpy()

print("Arrays ready. X_train_array shape:", X_train_array.shape)

#The seed that are the same as the baseline prediction
SEEDS = [3, 6, 9, 12, 15, 18, 21, 24, 27, 30]

#Runs without the class imbalance(SMOTE)
def run_no_smote(seed):
    model = eLCS(random_state=seed)
    start = time.perf_counter()
    model.fit(X_train_array, y_train_array)
    elapsed = time.perf_counter() - start
    preds = model.predict(X_test_array)
    probs = model.predict_proba(X_test_array)[:, 1]
    cm = confusion_matrix(y_test_array, preds)
    tn, fp, fn, tp = cm.ravel()
    return {
        "Seed": seed,
        "Accuracy": accuracy_score(y_test_array, preds),
        "Precision": precision_score(y_test_array, preds, zero_division=0),
        "Recall": recall_score(y_test_array, preds, zero_division=0),
        "F1": f1_score(y_test_array, preds, zero_division=0),
        "Balanced accuracy": balanced_accuracy_score(y_test_array, preds),
        "ROC-AUC": roc_auc_score(y_test_array, probs),
        "PR-AUC": average_precision_score(y_test_array, probs),
        "Training seconds": elapsed,
        "True negatives": tn, "False positives": fp,
        "False negatives": fn, "True positives": tp,
    }
#Runs with the class imbalance implemented(SMOTE)
def run_smote(seed):
    smote = SMOTE(random_state=seed)
    X_res, y_res = smote.fit_resample(X_train_array, y_train_array)
    model = eLCS(random_state=seed)
    start = time.perf_counter()
    model.fit(X_res, y_res)
    elapsed = time.perf_counter() - start
    preds = model.predict(X_test_array)
    probs = model.predict_proba(X_test_array)[:, 1]
    cm = confusion_matrix(y_test_array, preds)
    tn, fp, fn, tp = cm.ravel()
    return {
        "Seed": seed,
        "Accuracy": accuracy_score(y_test_array, preds),
        "Precision": precision_score(y_test_array, preds, zero_division=0),
        "Recall": recall_score(y_test_array, preds, zero_division=0),
        "F1": f1_score(y_test_array, preds, zero_division=0),
        "Balanced accuracy": balanced_accuracy_score(y_test_array, preds),
        "ROC-AUC": roc_auc_score(y_test_array, probs),
        "PR-AUC": average_precision_score(y_test_array, probs),
        "Training seconds": elapsed,
        "True negatives": tn, "False positives": fp,
        "False negatives": fn, "True positives": tp,
    }

no_smote_runs = [run_no_smote(s) for s in SEEDS]
no_smote_df = pd.DataFrame(no_smote_runs)
print("No SMOTE - 10 runs:")
display(no_smote_df)

smote_runs = [run_smote(s) for s in SEEDS]
smote_df = pd.DataFrame(smote_runs)
print("With SMOTE - 10 runs:")
display(smote_df)

Arrays ready. X_train_array shape: (222960, 35)
No SMOTE - 10 runs:


,Seed,Accuracy,Precision,Recall,F1,Balanced accuracy,ROC-AUC,PR-AUC,Training seconds,True negatives,False positives,False negatives,True positives
0,3,0.978867,0.000000,0.000000,0.000000,0.500000,0.623093,0.049519,46.606339,54563,0,1178,0
1,6,0.978867,0.000000,0.000000,0.000000,0.500000,0.644734,0.059376,37.705084,54563,0,1178,0
2,9,0.978795,0.300000,0.002547,0.005051,0.501209,0.597073,0.047600,38.149765,54556,7,1175,3
3,12,0.978813,0.000000,0.000000,0.000000,0.499973,0.570183,0.034534,37.867776,54560,3,1178,0
4,15,0.978867,0.000000,0.000000,0.000000,0.500000,0.562215,0.042381,38.193331,54563,0,1178,0
5,18,0.978867,0.000000,0.000000,0.000000,0.500000,0.553553,0.039653,36.964514,54563,0,1178,0
6,21,0.978867,0.000000,0.000000,0.000000,0.500000,0.575517,0.045619,38.334572,54563,0,1178,0
7,24,0.978867,0.500000,0.000849,0.001695,0.500415,0.611364,0.064501,37.987052,54562,1,1177,1
8,27,0.978490,0.200000,0.005942,0.011542,0.502715,0.577628,0.044593,36.887387,54535,28,1171,7
9,30,0.978185,0.104167,0.004244,0.008157,0.501728,0.562085,0.036606,37.665084,54520,43,1173,5


With SMOTE - 10 runs:


,Seed,Accuracy,Precision,Recall,F1,Balanced accuracy,ROC-AUC,PR-AUC,Training seconds,True negatives,False positives,False negatives,True positives
0,3,0.814714,0.100506,0.977080,0.182264,0.894144,0.931587,0.159612,82.404875,44262,10301,27,1151
1,6,0.791356,0.091016,0.987267,0.166667,0.887197,0.898956,0.099536,84.931392,42948,11615,15,1163
2,9,0.808992,0.097509,0.973684,0.177266,0.889560,0.929419,0.150306,1687.054069,43947,10616,31,1147
3,12,0.784180,0.088128,0.985569,0.161789,0.882701,0.932236,0.156425,83.899419,42550,12013,17,1161
4,15,0.786997,0.088938,0.982173,0.163107,0.882478,0.927689,0.155017,84.315346,42711,11852,21,1157
5,18,0.825209,0.104753,0.963497,0.188962,0.892861,0.936552,0.160211,80.947682,44863,9700,43,1135
6,21,0.792379,0.091359,0.986418,0.167230,0.887304,0.922829,0.132226,87.119622,43006,11557,16,1162
7,24,0.786638,0.089180,0.987267,0.163584,0.884787,0.926254,0.142481,87.745493,42685,11878,15,1163
8,27,0.823559,0.102926,0.952462,0.185777,0.886619,0.933953,0.157026,82.487264,44784,9779,56,1122
9,30,0.803412,0.094998,0.973684,0.173106,0.886710,0.934016,0.159611,81.282653,43636,10927,31,1147


### 4. Saved the output of the trainning

In [4]:
output_dir = PROJECT_ROOT / "outputs" / "improved_elcs_smote"
output_dir.mkdir(parents=True, exist_ok=True)

no_smote_df.to_csv(output_dir / "no_smote_runs.csv", index=False)
smote_df.to_csv(output_dir / "smote_runs.csv", index=False)

### 5. Summary of the output

In [5]:
summary_cols = ["Accuracy", "Precision", "Recall", "F1", "Balanced accuracy",
                "ROC-AUC", "PR-AUC", "Training seconds"]

no_smote_summary = (
    no_smote_df[summary_cols]
    .agg(["mean", "std", "min", "max"])
    .transpose()
    .reset_index()
    .rename(columns={"index": "Metric"})
)

smote_summary = (
    smote_df[summary_cols]
    .agg(["mean", "std", "min", "max"])
    .transpose()
    .reset_index()
    .rename(columns={"index": "Metric"})
)

print("No SMOTE summary (10 runs):")
display(no_smote_summary)

print("\nWith SMOTE summary (10 runs):")
display(smote_summary)

No SMOTE summary (10 runs):


,Metric,mean,std,min,max
0,Accuracy,0.978748,0.000230,0.978185,0.978867
1,Precision,0.110417,0.172862,0.000000,0.500000
2,Recall,0.001358,0.002162,0.000000,0.005942
3,F1,0.002644,0.004192,0.000000,0.011542
4,Balanced accuracy,0.500604,0.000963,0.499973,0.502715
5,ROC-AUC,0.587744,0.030151,0.553553,0.644734
6,PR-AUC,0.046438,0.009484,0.034534,0.064501
7,Training seconds,38.636091,2.842566,36.887387,46.606339



With SMOTE summary (10 runs):


,Metric,mean,std,min,max
0,Accuracy,0.801744,0.015634,0.784180,0.825209
1,Precision,0.094931,0.006160,0.088128,0.104753
2,Recall,0.976910,0.011576,0.952462,0.987267
3,F1,0.172975,0.010043,0.161789,0.188962
4,Balanced accuracy,0.887436,0.003862,0.882478,0.894144
5,ROC-AUC,0.927349,0.010785,0.898956,0.936552
6,PR-AUC,0.147245,0.018997,0.099536,0.160211
7,Training seconds,244.218781,506.965706,80.947682,1687.054069
